# PyTorch Dataset 与 DataLoader 数据加载

> 本笔记本是 [数据集操作.ipynb](./数据集操作.ipynb) 的 **PyTorch 等价版本**，
> 原版使用 TensorFlow tf.data API 构建数据流水线，本版使用 PyTorch Dataset 与 DataLoader 实现相同功能。

本教程演示如何使用 PyTorch 的 `torch.utils.data.Dataset` 和 `DataLoader` 构建高效的数据加载流水线，
并以 CSV 文件为例展示完整的数据处理流程。

## 学习目标

1. 掌握 `torch.utils.data.Dataset` 的核心接口（`__init__`、`__len__`、`__getitem__`）
2. 理解 `DataLoader` 的批处理与并行加载机制（`batch_size`、`shuffle`、`num_workers`）
3. 学会构建 CSV 数据流水线（pandas → TensorDataset → DataLoader）
4. 对比 TensorFlow `tf.data.Dataset` 与 PyTorch `DataLoader` 的设计差异

## 1. 环境配置

In [ ]:
import io
import os
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import (
    DataLoader,
    Dataset,
    SubsetRandomSampler,
    TensorDataset,
    WeightedRandomSampler,
    random_split,
)

# 设置随机种子确保结果可复现
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# 检测并选择计算设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch版本: {torch.__version__}")
print(f"计算设备: {device}")

## 2. torch.utils.data.Dataset 基础

### 2.1 Dataset 抽象类

`torch.utils.data.Dataset` 是 PyTorch 中所有数据集的抽象基类。
自定义数据集必须继承该类并实现以下三个方法：

| 方法 | 说明 |
|------|------|
| `__init__(self, ...)` | 初始化数据集，加载数据或建立索引 |
| `__len__(self)` | 返回数据集样本总数 |
| `__getitem__(self, idx)` | 根据索引 idx 返回单个样本 |

In [ ]:
"""
演示自定义 Dataset 的最小实现
Demonstrate a minimal custom Dataset implementation.
"""


class SimpleDataset(Dataset):
    """
    最简单的自定义数据集示例
    A minimal custom Dataset example.

    Parameters:
    -----------
    data : list or array-like
        样本数据列表 / List of sample data
    labels : list or array-like
        标签列表 / List of labels
    """

    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        """返回数据集大小 / Return dataset size"""
        return len(self.data)

    def __getitem__(self, idx):
        """根据索引获取样本 / Get sample by index"""
        return self.data[idx], self.labels[idx]


# 创建简单数据集
simple_data = list(range(10))
simple_labels = [x * 2 for x in simple_data]

dataset = SimpleDataset(simple_data, simple_labels)

print(f"数据集大小: {len(dataset)}")
print(f"第0个样本: {dataset[0]}")
print(f"第5个样本: {dataset[5]}")
print(f"切片访问（需要自行实现）: {[dataset[i] for i in range(3)]}")

### 2.2 TensorDataset 快捷方式

当数据已经是张量形式时，可以使用 `TensorDataset` 快速创建数据集，
无需手写 `__init__`、`__len__`、`__getitem__`。

`TensorDataset` 会将传入的多个张量沿第一个维度进行配对。

In [ ]:
"""
使用 TensorDataset 快速创建数据集
Create a dataset quickly using TensorDataset.
"""

# 创建特征和标签张量
features = torch.randn(100, 5)   # 100个样本，5个特征
labels = torch.randint(0, 3, (100,))  # 100个标签（0/1/2三分类）

# 使用 TensorDataset
tensor_dataset = TensorDataset(features, labels)

print(f"数据集大小: {len(tensor_dataset)}")
print(f"第一个样本（特征形状, 标签）: ({tensor_dataset[0][0].shape}, {tensor_dataset[0][1]})")

# TensorDataset 支持索引访问
sample_features, sample_label = tensor_dataset[0]
print("\n索引访问示例:")
print(f"  特征: {sample_features}")
print(f"  标签: {sample_label}")

# 也可以同时获取多个张量
tensor_dataset_3 = TensorDataset(features, labels, torch.randn(100))  # 第三个张量：权重
f, l, w = tensor_dataset_3[0]
print(f"\n三张量数据集访问: 特征形状={f.shape}, 标签={l.item()}, 权重={w.item():.4f}")

### 2.3 自定义 Dataset 类

在实际项目中，我们通常需要自定义 Dataset 类来处理更复杂的数据逻辑，
例如数据预处理、特征工程、多模态数据加载等。

下面是一个完整的自定义 Dataset 示例，使用合成数据模拟真实场景。

In [ ]:
"""
完整的自定义 Dataset 类示例
A complete custom Dataset class example.
"""


class CustomDataset(Dataset):
    """
    自定义数据集类，支持数据预处理和类型转换
    Custom Dataset class with preprocessing and type conversion.

    Parameters:
    -----------
    features : numpy.ndarray
        特征矩阵 / Feature matrix
    labels : numpy.ndarray
        标签数组 / Label array
    transform : callable, optional
        特征变换函数 / Feature transform function
    target_transform : callable, optional
        标签变换函数 / Label transform function
    """

    def __init__(self, features, labels, transform=None, target_transform=None):
        """
        初始化数据集
        Initialize the dataset.
        """
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        """返回数据集大小 / Return dataset size"""
        return len(self.features)

    def __getitem__(self, idx):
        """
        根据索引获取样本，并应用变换
        Get sample by index and apply transforms.
        """
        x = self.features[idx]
        y = self.labels[idx]

        if self.transform:
            x = self.transform(x)
        if self.target_transform:
            y = self.target_transform(y)

        return x, y


# 创建合成数据
np.random.seed(RANDOM_SEED)
synthetic_features = np.random.randn(500, 8).astype(np.float32)
synthetic_labels = (synthetic_features[:, 0] + synthetic_features[:, 1] > 0).astype(np.int64)

# 定义变换函数
def normalize_feature(x):
    """简单的标准化变换 / Simple normalization transform"""
    return (x - x.mean()) / (x.std() + 1e-8)

# 创建自定义数据集（带变换）
custom_ds = CustomDataset(
    synthetic_features, synthetic_labels,
    transform=normalize_feature
)

print(f"数据集大小: {len(custom_ds)}")
print(f"特征形状: {custom_ds[0][0].shape}")
print(f"样本示例: 特征前3={custom_ds[0][0][:3]}, 标签={custom_ds[0][1]}")
print(f"类别分布: 类0={sum(synthetic_labels==0)}, 类1={sum(synthetic_labels==1)}")

## 3. torch.utils.data.DataLoader 详解

`DataLoader` 是 PyTorch 中负责批处理、打乱、并行加载的核心类。
它将 `Dataset` 包装成一个可迭代对象，自动处理批次划分和数据打乱。

### 3.1 基本参数: batch_size, shuffle

In [ ]:
"""
DataLoader 基本参数演示：batch_size 和 shuffle
Demonstrate DataLoader basic parameters: batch_size and shuffle.
"""

# 使用前面的 TensorDataset
features_small = torch.arange(10, dtype=torch.float32).unsqueeze(1)  # [0,1,...,9]
labels_small = torch.arange(10, dtype=torch.float32) * 10  # [0,10,...,90]
small_dataset = TensorDataset(features_small, labels_small)

# 不同 batch_size 的效果
print("=== 不同 batch_size ===")
for bs in [2, 3, 4]:
    loader = DataLoader(small_dataset, batch_size=bs, shuffle=False)
    print(f"\nbatch_size={bs}, 总批次={len(loader)}")
    for i, (batch_x, batch_y) in enumerate(loader):
        print(f"  批次{i}: x={batch_x.squeeze().tolist()}, y={batch_y.tolist()}")

# shuffle 的效果
print("\n=== shuffle 对比 ===")
loader_no_shuffle = DataLoader(small_dataset, batch_size=3, shuffle=False)
loader_shuffle = DataLoader(small_dataset, batch_size=3, shuffle=True)

print("shuffle=False:", [batch[0].squeeze().tolist() for batch in loader_no_shuffle])
print("shuffle=True:", [batch[0].squeeze().tolist() for batch in loader_shuffle])
# 注意：每次迭代 shuffle=True 的顺序不同

### 3.2 num_workers 并行加载

`num_workers` 控制数据加载的子进程数量：
- `num_workers=0`（默认）：在主进程中加载数据，简单但可能成为瓶颈
- `num_workers>0`：使用多进程并行加载，可显著提升 I/O 密集型任务的效率

一般建议：`num_workers` 设为 CPU 核心数的 2~4 倍，但需要根据实际硬件和数据量调整。

In [ ]:
"""
num_workers 并行加载性能对比
Compare performance with different num_workers values.
"""

# 创建较大的数据集用于计时
large_features = torch.randn(5000, 100)
large_labels = torch.randint(0, 10, (5000,))
large_dataset = TensorDataset(large_features, large_labels)

print(f"数据集大小: {len(large_dataset)}")
print(f"CPU核心数: {os.cpu_count()}")
print()

# 测试不同 num_workers 的加载时间
for nw in [0, 2, 4]:
    loader = DataLoader(
        large_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=nw,
    )
    start = time.time()
    for batch_x, batch_y in loader:
        pass  # 仅迭代，不做计算
    elapsed = time.time() - start
    print(f"num_workers={nw}: 迭代耗时 {elapsed:.4f}s ({len(loader)} 批次)")

# 注意：小数据集在内存中时，num_workers 的优势不明显
# 真正的加速体现在磁盘I/O密集的场景（如读取大量图片文件）

### 3.3 其他常用参数

| 参数 | 说明 |
|------|------|
| `drop_last` | 当样本数不能被 batch_size 整除时，是否丢弃最后一个不完整批次 |
| `pin_memory` | 将数据固定在内存中，加速 GPU 传输（建议 GPU 训练时设为 True） |
| `sampler` | 自定义采样策略（与 shuffle 互斥） |

In [ ]:
"""
DataLoader 其他常用参数演示
Demonstrate other commonly used DataLoader parameters.
"""

# drop_last：是否丢弃最后不完整批次
loader_with_last = DataLoader(small_dataset, batch_size=3, shuffle=False, drop_last=False)
loader_drop_last = DataLoader(small_dataset, batch_size=3, shuffle=False, drop_last=True)

print(f"数据集大小: {len(small_dataset)}, batch_size=3")
print(f"drop_last=False: {len(loader_with_last)} 批次")
print(f"drop_last=True:  {len(loader_drop_last)} 批次")

# 查看最后一个批次
last_batch_keep = list(loader_with_last)[-1]
last_batch_drop = list(loader_drop_last)[-1]
print(f"\ndrop_last=False 最后批次大小: {last_batch_keep[0].shape[0]}")
print(f"drop_last=True  最后批次大小: {last_batch_drop[0].shape[0]}")

# pin_memory：GPU 训练时建议开启
loader_pinned = DataLoader(
    large_dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=True,  # 将数据固定在页锁定内存中，加速 CPU->GPU 传输
)
print("\npin_memory=True 的 DataLoader 已创建")
print(f"批次形状: {next(iter(loader_pinned))[0].shape}")

### 3.4 迭代 DataLoader

DataLoader 是一个可迭代对象，有两种常见的迭代方式：
1. `for batch in dataloader:` —— 标准循环方式
2. `next(iter(dataloader))` —— 获取单个批次（调试时常用）

In [ ]:
"""
DataLoader 迭代方式演示
Demonstrate DataLoader iteration patterns.
"""

demo_loader = DataLoader(small_dataset, batch_size=3, shuffle=False)

# 方式1：next(iter(dataloader)) —— 获取第一个批次
first_batch = next(iter(demo_loader))
print("方式1: next(iter(dataloader))")
print(f"  特征: {first_batch[0].squeeze().tolist()}")
print(f"  标签: {first_batch[1].tolist()}")

# 方式2：for 循环迭代
print("\n方式2: for batch in dataloader")
for i, (batch_x, batch_y) in enumerate(demo_loader):
    print(f"  批次{i}: x={batch_x.squeeze().tolist()}, y={batch_y.tolist()}")

# 注意：DataLoader 每次迭代都从数据集重新采样
# 如果 shuffle=True，每次 for 循环的顺序可能不同
shuffle_loader = DataLoader(small_dataset, batch_size=3, shuffle=True)
print("\nshuffle=True 时两次迭代对比:")
order1 = [b[0].squeeze().tolist() for b in shuffle_loader]
order2 = [b[0].squeeze().tolist() for b in shuffle_loader]
print(f"  第1次: {order1}")
print(f"  第2次: {order2}")

## 4. CSV 数据流水线实战

### 4.1 pandas 读取 CSV

在实际项目中，CSV 是最常见的数据格式之一。
我们首先用 pandas 读取 CSV 数据，然后转换为 PyTorch 张量。

In [ ]:
"""
创建示例 CSV 数据并用 pandas 读取
Create sample CSV data and read with pandas.
"""

# 在内存中创建示例 CSV 数据
np.random.seed(RANDOM_SEED)
n_samples = 200

csv_data = pd.DataFrame({
    'age': np.random.randint(18, 70, n_samples),
    'income': np.random.uniform(20000, 120000, n_samples).round(2),
    'education_years': np.random.randint(8, 20, n_samples),
    'work_hours': np.random.randint(20, 60, n_samples),
    'credit_score': np.random.randint(300, 850, n_samples),
    'loan_amount': np.random.uniform(1000, 50000, n_samples).round(2),
})

# 生成标签：基于特征的简单规则
csv_data['default'] = (
    (csv_data['credit_score'] < 500) & (csv_data['loan_amount'] > 20000)
).astype(int)

# 保存为 CSV 字符串（模拟从文件读取）
csv_buffer = io.StringIO()
csv_data.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

# 用 pandas 读取
df = pd.read_csv(csv_buffer)

print(f"数据形状: {df.shape}")
print("\n前5行:")
df.head()

### 4.2 pandas → TensorDataset → DataLoader

这是最简洁的 CSV 数据流水线：

```
CSV文件 → pandas DataFrame → numpy数组 → torch张量 → TensorDataset → DataLoader
```

In [ ]:
"""
完整的 pandas → TensorDataset → DataLoader 流水线
Full pipeline: pandas → TensorDataset → DataLoader.
"""

# Step 1: 分离特征和标签
feature_cols = ['age', 'income', 'education_years', 'work_hours', 'credit_score', 'loan_amount']
X = df[feature_cols].values          # numpy array (200, 6)
y = df['default'].values             # numpy array (200,)

print(f"特征形状: {X.shape}, 标签形状: {y.shape}")
print(f"类别分布: 类0={sum(y==0)}, 类1={sum(y==1)}")

# Step 2: 转换为 PyTorch 张量
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# Step 3: 创建 TensorDataset
csv_dataset = TensorDataset(X_tensor, y_tensor)

# Step 4: 创建 DataLoader
csv_loader = DataLoader(csv_dataset, batch_size=16, shuffle=True)

print("\nDataLoader 信息:")
print(f"  数据集大小: {len(csv_dataset)}")
print("  批次大小: 16")
print(f"  总批次数: {len(csv_loader)}")

# 验证一个批次
batch_x, batch_y = next(iter(csv_loader))
print("\n一个批次示例:")
print(f"  特征形状: {batch_x.shape}")
print(f"  标签形状: {batch_y.shape}")
print(f"  标签值: {batch_y.tolist()}")

### 4.3 自定义 CSVDataset 类

对于更复杂的场景（如需要列选择、数据清洗、特征变换等），
可以创建自定义的 CSVDataset 类，将所有数据逻辑封装在一起。

In [ ]:
"""
自定义 CSVDataset 类
Custom CSVDataset class for CSV file handling.
"""


class CSVDataset(Dataset):
    """
    从 CSV 文件或 DataFrame 创建的自定义数据集
    Custom Dataset for loading data from CSV files or DataFrames.

    Parameters:
    -----------
    data : str or pd.DataFrame
        CSV文件路径或 pandas DataFrame / CSV file path or pandas DataFrame
    feature_cols : list of str
        特征列名列表 / List of feature column names
    label_col : str
        标签列名 / Label column name
    transform : callable, optional
        特征变换函数 / Feature transform function
    """

    def __init__(self, data, feature_cols, label_col, transform=None):
        """
        初始化数据集
        Initialize the dataset.
        """
        # 支持文件路径或 DataFrame
        if isinstance(data, str):
            self.df = pd.read_csv(data)
        else:
            self.df = data.copy()

        self.feature_cols = feature_cols
        self.label_col = label_col
        self.transform = transform

        # 预处理：提取特征和标签
        self.features = self.df[feature_cols].values.astype(np.float32)
        self.labels = self.df[label_col].values.astype(np.int64)

    def __len__(self):
        """返回数据集大小 / Return dataset size"""
        return len(self.features)

    def __getitem__(self, idx):
        """
        获取样本，并应用变换
        Get sample by index and apply transform.
        """
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)

        if self.transform:
            x = self.transform(x)

        return x, y

    def get_feature_names(self):
        """返回特征列名 / Return feature column names"""
        return self.feature_cols


# 使用自定义 CSVDataset
custom_csv_ds = CSVDataset(
    data=df,
    feature_cols=feature_cols,
    label_col='default',
)

print(f"自定义 CSVDataset 大小: {len(custom_csv_ds)}")
print(f"特征列: {custom_csv_ds.get_feature_names()}")
x_sample, y_sample = custom_csv_ds[0]
print(f"样本0: 特征={x_sample.tolist()}, 标签={y_sample.item()}")

# 通过 DataLoader 加载
custom_csv_loader = DataLoader(custom_csv_ds, batch_size=32, shuffle=True)
batch_x, batch_y = next(iter(custom_csv_loader))
print(f"\nDataLoader 批次: 特征形状={batch_x.shape}, 标签形状={batch_y.shape}")

### 4.4 完整训练流水线示例

下面展示一个完整的训练流水线：
数据拆分 → 标准化 → 创建 DataLoader → 构建模型 → 训练循环。

In [ ]:
"""
完整训练流水线：数据拆分与标准化
Full training pipeline: data split and standardization.
"""

# Step 1: 数据拆分
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=RANDOM_SEED, stratify=y_train
)

print(f"训练集: {X_train.shape[0]}, 验证集: {X_val.shape[0]}, 测试集: {X_test.shape[0]}")

# Step 2: 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Step 3: 创建 TensorDataset 和 DataLoader
BATCH_SIZE = 32

train_ds = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)
val_ds = TensorDataset(
    torch.tensor(X_val_scaled, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long),
)
test_ds = TensorDataset(
    torch.tensor(X_test_scaled, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"训练 DataLoader: {len(train_loader)} 批次")
print(f"验证 DataLoader: {len(val_loader)} 批次")
print(f"测试 DataLoader: {len(test_loader)} 批次")

In [ ]:
"""
构建模型并训练
Build model and train.
"""

# Step 4: 构建简单分类模型
class SimpleClassifier(nn.Module):
    """
    简单的二分类 MLP 模型
    A simple binary classification MLP model.

    Parameters:
    -----------
    input_dim : int
        输入特征维度 / Input feature dimension
    hidden_dim : int
        隐藏层维度 / Hidden layer dimension
    """

    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),  # 二分类输出
        )

    def forward(self, x):
        return self.net(x)


model = SimpleClassifier(input_dim=6, hidden_dim=32).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"模型结构:\n{model}")
print(f"参数总数: {sum(p.numel() for p in model.parameters())}")

In [ ]:
"""
训练循环
Training loop.
"""

EPOCHS = 30
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    # 训练阶段
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        correct += (outputs.argmax(1) == y_batch).sum().item()
        total += X_batch.size(0)

    train_loss = total_loss / total
    train_acc = correct / total

    # 验证阶段
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            total_loss += criterion(outputs, y_batch).item() * X_batch.size(0)
            correct += (outputs.argmax(1) == y_batch).sum().item()
            total += X_batch.size(0)

    val_loss = total_loss / total
    val_acc = correct / total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} - "
              f"loss: {train_loss:.4f} - acc: {train_acc:.4f} - "
              f"val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")

# 测试集评估
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        correct += (outputs.argmax(1) == y_batch).sum().item()
        total += X_batch.size(0)
print(f"\n测试集准确率: {correct/total:.4f}")

## 5. 数据集拆分与采样

### 5.1 random_split

`torch.utils.data.random_split` 可以按比例随机拆分数据集，
这是最简单的训练/验证拆分方式。

In [ ]:
"""
使用 random_split 拆分数据集
Split dataset using random_split.
"""

# 创建完整数据集
full_features = torch.randn(1000, 10)
full_labels = torch.randint(0, 5, (1000,))
full_dataset = TensorDataset(full_features, full_labels)

# 按 80%/10%/10% 拆分
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_subset, val_subset, test_subset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)

print(f"原始数据集大小: {len(full_dataset)}")
print(f"训练子集: {len(train_subset)}")
print(f"验证子集: {len(val_subset)}")
print(f"测试子集: {len(test_subset)}")

# random_split 返回的是 Subset 对象，可以直接用 DataLoader 加载
train_loader_split = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader_split = DataLoader(val_subset, batch_size=32, shuffle=False)

batch_x, batch_y = next(iter(train_loader_split))
print(f"\n训练 DataLoader 批次: 特征形状={batch_x.shape}, 标签形状={batch_y.shape}")

### 5.2 SubsetRandomSampler

`SubsetRandomSampler` 通过索引采样实现数据集拆分，
可以更灵活地控制哪些样本进入训练集或验证集。

与 `random_split` 不同，`SubsetRandomSampler` 不创建数据集副本，
而是通过索引列表在同一个数据集上采样。

In [ ]:
"""
使用 SubsetRandomSampler 拆分数据集
Split dataset using SubsetRandomSampler.
"""

# 创建索引列表
num_samples = len(full_dataset)
indices = list(range(num_samples))
np.random.shuffle(indices)

# 划分索引
train_idx = indices[:800]
val_idx = indices[800:900]
test_idx = indices[900:]

# 创建 Sampler
train_sampler = SubsetRandomSampler(train_idx)
val_sampler = SubsetRandomSampler(val_idx)
test_sampler = SubsetRandomSampler(test_idx)

# 使用同一个数据集，不同的 Sampler
train_loader_sampler = DataLoader(full_dataset, batch_size=32, sampler=train_sampler)
val_loader_sampler = DataLoader(full_dataset, batch_size=32, sampler=val_sampler)
test_loader_sampler = DataLoader(full_dataset, batch_size=32, sampler=test_sampler)

print(f"训练 DataLoader 批次数: {len(train_loader_sampler)}")
print(f"验证 DataLoader 批次数: {len(val_loader_sampler)}")
print(f"测试 DataLoader 批次数: {len(test_loader_sampler)}")

# 注意：使用 sampler 时不能同时设置 shuffle=True
# train_loader = DataLoader(full_dataset, batch_size=32, sampler=train_sampler, shuffle=True)
# 上述代码会报错：sampler 和 shuffle 互斥

### 5.3 WeightedRandomSampler

当数据集类别不平衡时，可以使用 `WeightedRandomSampler` 进行过采样，
让少数类样本被更频繁地选中，从而平衡训练过程中的类别分布。

In [ ]:
"""
使用 WeightedRandomSampler 处理类别不平衡
Handle class imbalance with WeightedRandomSampler.
"""

# 创建不平衡数据集：类0有900个，类1有100个
np.random.seed(RANDOM_SEED)
imbalanced_features = torch.randn(1000, 5)
imbalanced_labels = torch.cat([torch.zeros(900, dtype=torch.long),
                                torch.ones(100, dtype=torch.long)])
imbalanced_dataset = TensorDataset(imbalanced_features, imbalanced_labels)

print(f"不平衡数据集: 类0={sum(imbalanced_labels==0).item()}, 类1={sum(imbalanced_labels==1).item()}")

# 计算每个样本的权重
class_counts = torch.bincount(imbalanced_labels)
class_weights = 1.0 / class_counts.float()  # 类别权重：数量越少权重越大
sample_weights = class_weights[imbalanced_labels]  # 每个样本的权重

print(f"\n类别数量: {class_counts.tolist()}")
print(f"类别权重: {class_weights.tolist()}")
print(f"样本权重示例（前5个）: {sample_weights[:5].tolist()}")

# 创建 WeightedRandomSampler
weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(imbalanced_dataset),  # 每个epoch采样的总数
    replacement=True,  # 允许重复采样
)

# 使用加权采样的 DataLoader
balanced_loader = DataLoader(imbalanced_dataset, batch_size=32, sampler=weighted_sampler)

# 统计一个 epoch 中各类别的采样数量
class_0_count, class_1_count = 0, 0
for _, batch_y in balanced_loader:
    class_0_count += (batch_y == 0).sum().item()
    class_1_count += (batch_y == 1).sum().item()

print("\n加权采样后一个epoch的类别分布:")
print(f"  类0: {class_0_count}, 类1: {class_1_count}")
print(f"  比例: {class_0_count/class_1_count:.2f}:1 (原始为 9:1)")

## 小结

### Dataset 核心概念

| 概念 | 说明 |
|------|------|
| `Dataset` | 抽象基类，需实现 `__init__`、`__len__`、`__getitem__` |
| `TensorDataset` | 快捷方式，直接从张量创建数据集 |
| 自定义 Dataset | 封装数据加载、预处理、变换等逻辑 |

### DataLoader 核心概念

| 参数 | 说明 |
|------|------|
| `batch_size` | 每个批次的样本数 |
| `shuffle` | 是否每个 epoch 打乱数据顺序 |
| `num_workers` | 数据加载的子进程数 |
| `drop_last` | 是否丢弃最后不完整批次 |
| `pin_memory` | 是否使用页锁定内存加速 GPU 传输 |
| `sampler` | 自定义采样策略 |

### 数据集拆分与采样

| 方法 | 适用场景 |
|------|----------|
| `random_split` | 简单按比例拆分 |
| `SubsetRandomSampler` | 灵活索引采样，不复制数据 |
| `WeightedRandomSampler` | 处理类别不平衡 |

### 最佳实践

1. 训练集使用 `shuffle=True`，验证/测试集使用 `shuffle=False`
2. GPU 训练时设置 `pin_memory=True`
3. I/O 密集型任务适当增大 `num_workers`
4. 类别不平衡时使用 `WeightedRandomSampler` 或加权损失函数
5. 标准化参数只从训练集计算，再应用到验证/测试集

## TF vs PyTorch 对照

| 概念 | TensorFlow tf.data | PyTorch DataLoader |
|------|-------------------|---------------------|
| 数据集创建 | `tf.data.Dataset.from_tensor_slices((X, y))` | `TensorDataset(X, y)` |
| 自定义数据集 | 无需继承，直接从数据创建 | 继承 `Dataset`，实现 `__getitem__` |
| 数据变换 | `dataset.map(transform_fn)` | 在 `Dataset.__getitem__` 中实现 |
| 打乱 | `dataset.shuffle(buffer_size=1000)` | `DataLoader(dataset, shuffle=True)` |
| 批处理 | `dataset.batch(32)` | `DataLoader(dataset, batch_size=32)` |
| 重复 | `dataset.repeat(num_epochs)` | DataLoader 自动重复（for 循环控制 epoch） |
| 预取 | `dataset.prefetch(tf.data.AUTOTUNE)` | `DataLoader(num_workers=N, prefetch_factor=2)` |
| 并行映射 | `dataset.map(fn, num_parallel_calls=AUTOTUNE)` | `DataLoader(num_workers=N)` |
| 迭代方式 | `for batch in dataset:` | `for batch in dataloader:` |
| 获取单批次 | `next(iter(dataset))` | `next(iter(dataloader))` |
| 过滤 | `dataset.filter(predicate)` | 需在 `__getitem__` 或 collate_fn 中实现 |
| 拼接 | `dataset1.concatenate(dataset2)` | `ConcatDataset([ds1, ds2])` |
| 数据集拆分 | `dataset.take(N)` / `dataset.skip(N)` | `random_split(dataset, [train, val])` |
| 类别平衡 | 无内置采样器 | `WeightedRandomSampler` |
| 流水线风格 | 链式调用 `.map().shuffle().batch().prefetch()` | Dataset + DataLoader 分离式设计 |
| 设计理念 | 声明式流水线（惰性求值） | 命令式迭代（即时求值） |

## 练习

### 练习1：创建自定义 Dataset 处理 JSON 数据

创建一个 `JSONDataset` 类，能够从 JSON 文件或字典列表加载数据：
```python
import json

class JSONDataset(Dataset):
    def __init__(self, data, feature_keys, label_key):
        # 支持文件路径或列表
        if isinstance(data, str):
            with open(data, 'r') as f:
                self.records = json.load(f)
        else:
            self.records = data
        self.feature_keys = feature_keys
        self.label_key = label_key

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        features = torch.tensor([record[k] for k in self.feature_keys], dtype=torch.float32)
        label = torch.tensor(record[self.label_key], dtype=torch.long)
        return features, label
```
思考：JSON 数据与 CSV 数据在加载方式上有什么区别？如何处理嵌套 JSON 结构？

### 练习2：对比不同 num_workers 的性能

创建一个从磁盘读取文件的 Dataset（如模拟图片加载），
对比 `num_workers=0, 2, 4, 8` 时的数据加载速度：
```python
class FileDataset(Dataset):
    def __init__(self, file_dir):
        self.file_list = sorted(os.listdir(file_dir))
        self.file_dir = file_dir

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        # 模拟磁盘I/O延迟
        time.sleep(0.01)
        data = torch.randn(3, 224, 224)  # 模拟图片张量
        return data

# 对比不同 num_workers
for nw in [0, 2, 4, 8]:
    loader = DataLoader(FileDataset('.'), batch_size=16, num_workers=nw)
    start = time.time()
    for _ in loader:
        pass
    print(f"num_workers={nw}: {time.time()-start:.2f}s")
```
思考：为什么磁盘I/O密集型任务中 `num_workers` 的加速效果更明显？是否存在最优值？

### 练习3：实现平衡批次采样器

实现一个 `BalancedBatchSampler`，确保每个批次中各类别的样本数量大致相等：
```python
class BalancedBatchSampler(Sampler):
    def __init__(self, labels, batch_size):
        self.labels = np.array(labels)
        self.batch_size = batch_size
        self.n_classes = len(np.unique(labels))
        self.samples_per_class = batch_size // self.n_classes

    def __iter__(self):
        # 为每个类别创建索引列表并打乱
        class_indices = []
        for c in range(self.n_classes):
            idx = np.where(self.labels == c)[0]
            np.random.shuffle(idx)
            class_indices.append(idx.tolist())

        # 从每个类别中轮流取样本组成批次
        while all(len(ci) >= self.samples_per_class for ci in class_indices):
            batch = []
            for ci in class_indices:
                batch.extend(ci[:self.samples_per_class])
                del ci[:self.samples_per_class]
            yield batch

    def __len__(self):
        return len(self.labels) // self.batch_size
```
思考：`BalancedBatchSampler` 与 `WeightedRandomSampler` 在处理类别不平衡时有什么区别？各有什么优缺点？